# Grassmannian Chain Theory (GCT) for Physics

This notebook explores the GCT (Grassmannian Chain Theory) module, which provides a geometric framework for understanding gauge-gravity transitions in physics.

## Overview

GCT models the Standard Model and gravity as arising from a chain of Grassmannian manifolds:

$$\text{Gr}(3,16) \to \text{Gr}(3,13) \to \text{Gr}(3,10) \to \text{Gr}(3,7) \to \text{Gr}(3,4)$$

Each transition corresponds to dimensional reduction, with the final step connecting to 4D spacetime.

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

## 1. The GCT Chain

The chain consists of 5 Grassmannians with fiber dimension k=3.

In [ ]:
from grasscalc.gct.chain import GCT_CHAIN, total_gct_dimension, gct_summary

# Display the chain
print("GCT Chain Structure")
print("=" * 50)
print(f"{'Manifold':<15} {'k':<5} {'n':<5} {'dim Gr(k,n)':<12}")
print("-" * 50)

for gr in GCT_CHAIN:
    dim = gr['k'] * (gr['n'] - gr['k'])
    print(f"Gr({gr['k']},{gr['n']})" + " " * 7 + f"{gr['k']:<5} {gr['n']:<5} {dim:<12}")

print("-" * 50)
print(f"Total dimension: {total_gct_dimension()}")

In [ ]:
# Full summary
print(gct_summary())

## 2. The Weinberg Angle

A key prediction of GCT is the Weinberg angle (weak mixing angle):

$$\sin^2\theta_W = \frac{3}{13} \approx 0.23077$$

This arises from the ratio 3/13 in the Gr(3,16) → Gr(3,13) transition.

The experimental value at the Z pole is approximately 0.23122.

In [ ]:
from grasscalc.gct.weinberg import (
    sin2_weinberg_exact, sin2_weinberg_numeric,
    WEINBERG_EXPERIMENTAL, weinberg_error
)

print("Weinberg Angle Analysis")
print("=" * 50)
print(f"GCT prediction (exact):    sin²θ_W = 3/13")
print(f"GCT prediction (numeric):  sin²θ_W = {sin2_weinberg_numeric():.8f}")
print(f"Experimental value:        sin²θ_W = {WEINBERG_EXPERIMENTAL:.6f} ± 0.00016")
print(f"\nRelative error: {weinberg_error() * 100:.4f}%")

In [ ]:
# Visualize the comparison
import matplotlib.pyplot as plt

gct_value = sin2_weinberg_numeric()
exp_value = WEINBERG_EXPERIMENTAL
exp_error = 0.00016

fig, ax = plt.subplots(figsize=(8, 4))

ax.errorbar([0], [exp_value], yerr=[exp_error], fmt='ro', markersize=10, 
            capsize=5, label=f'Experimental: {exp_value:.5f}')
ax.plot([0], [gct_value], 'bs', markersize=12, 
        label=f'GCT (3/13): {gct_value:.5f}')

ax.axhline(y=gct_value, color='blue', linestyle='--', alpha=0.3)
ax.axhline(y=exp_value, color='red', linestyle='--', alpha=0.3)

ax.set_ylabel(r'$\sin^2\theta_W$', fontsize=12)
ax.set_title('Weinberg Angle: GCT Prediction vs Experiment', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(-0.5, 0.5)
ax.set_ylim(0.229, 0.233)
ax.set_xticks([])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Gauge-Gravity Decomposition

The 7 Grassmannians in the full analysis decompose into:
- 4 gauge manifolds (Standard Model structure)
- 3 gravity manifolds (spacetime + extra dimensions)

In [ ]:
from grasscalc.gct.chain import gauge_gravity_decomposition

gauge, gravity = gauge_gravity_decomposition()

print("Gauge Sector (Standard Model)")
print("-" * 40)
for g in gauge:
    dim = g['k'] * (g['n'] - g['k'])
    print(f"  Gr({g['k']},{g['n']}): dim = {dim}")
print(f"  Total: {sum(g['k'] * (g['n'] - g['k']) for g in gauge)}")

print("\nGravity Sector")
print("-" * 40)
for g in gravity:
    dim = g['k'] * (g['n'] - g['k'])
    print(f"  Gr({g['k']},{g['n']}): dim = {dim}")
print(f"  Total: {sum(g['k'] * (g['n'] - g['k']) for g in gravity)}")

## 4. E8 Structure

The 248-dimensional E8 Lie algebra decomposes in a way consistent with the GCT framework.

In [ ]:
from grasscalc.gct.chain import e8_dimension, e8_halfspinor_dimension, e8_root_count

print("E8 Structure")
print("=" * 50)
print(f"E8 dimension:         {e8_dimension()}")
print(f"E8 root count:        {e8_root_count()}")
print(f"E8 half-spinor dim:   {e8_halfspinor_dimension()}")
print(f"\nDecomposition: 248 = 120 + 128")
print(f"  120 = so(16) adjoint")
print(f"  128 = half-spinor of so(16)")

## 5. Transition Operators

Transitions between Grassmannians in the chain can be modeled as operators on the fiber bundles.

In [ ]:
from grasscalc.gct.transitions import gct_transition_chain, TransitionType

transitions = gct_transition_chain()

print("GCT Transition Chain")
print("=" * 60)
print(f"{'Transition':<25} {'Type':<15} {'Δn':<5}")
print("-" * 60)

for t in transitions:
    source = f"Gr({t['source']['k']},{t['source']['n']})"
    target = f"Gr({t['target']['k']},{t['target']['n']})"
    name = f"{source} → {target}"
    delta_n = t['source']['n'] - t['target']['n']
    print(f"{name:<25} {t['type'].value:<15} {delta_n:<5}")

## 6. Sharp Bound in GCT Context

The Sharp Bound Theorem has special significance in GCT: transitions with $\Delta k \neq 0$ must saturate the bound.

In [ ]:
from grasscalc.core.random import sample_grassmann
from grasscalc.layer2.sharp_bound import verify_sharp_bound

rng = np.random.default_rng(42)

# Simulate a transition-like structure
print("Sharp Bound in GCT Transitions")
print("=" * 50)

# For same k (as in GCT), bound is 0 - but random subspaces have positive distance
for src, tgt in [(16, 13), (13, 10), (10, 7), (7, 4)]:
    k = 3
    
    # Sample subspaces
    U = sample_grassmann(k, src, rng)
    
    # Embed into lower dimension (take first tgt rows)
    U_proj = U[:tgt, :]
    Q, _ = np.linalg.qr(U_proj)  # Re-orthonormalize
    V = Q
    
    # Compute bound satisfaction
    bound = abs(k - k)  # Same k, so bound is 0
    
    print(f"Gr({k},{src}) → Gr({k},{tgt}): Δn = {src - tgt}, bound = {bound}")

## 7. Validation Tests

Run the full GCT validation suite to verify all theoretical constraints.

In [ ]:
from grasscalc.gct.validation import run_all_tests, test_weinberg_prediction

print("Running GCT Validation Suite")
print("=" * 50)

# Run all tests
passed, failed = run_all_tests()

print(f"\nResults: {passed} passed, {failed} failed")
if failed == 0:
    print("✓ All GCT theoretical constraints satisfied!")

In [ ]:
# Detailed Weinberg test
result = test_weinberg_prediction()
print(f"\nWeinberg angle test: {'PASSED' if result['passed'] else 'FAILED'}")
print(f"  Predicted: {result['predicted']:.6f}")
print(f"  Experimental: {result['experimental']:.6f}")
print(f"  Error: {result['error_percent']:.4f}%")

## Summary

The GCT module provides:

1. **Chain structure**: 5 Grassmannians modeling gauge and gravity sectors
2. **Weinberg angle**: Exact prediction sin²θ_W = 3/13 (0.2% error)
3. **Gauge-gravity split**: Natural decomposition into Standard Model + spacetime
4. **E8 connection**: Consistent with exceptional group structure
5. **Transitions**: Operators connecting scales
6. **Validation**: Comprehensive test suite for theoretical constraints

This geometric framework provides a unified perspective on the structure of fundamental physics.